# multilingual-e5-large + UMAP + Agglomerative (Ward) clustering

Pairs the finalized embedding model, `multilingual-e5-large`, with **Ward-linkage Agglomerative
clustering** instead of HDBSCAN. Motivated by news-event-detection research (arXiv:2406.10552)
finding agglomerative clustering more robust than HDBSCAN for this exact task.

Agglomerative clustering has no built-in "noise" concept and needs a target cluster count or
distance threshold — here we sweep `distance_threshold` and keep the cut that maximizes silhouette
score.

In [1]:
!pip install -q umap-learn

In [2]:
# Paths (Kaggle)
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")
RESULTS_DIR = Path("/kaggle/working/results/e5_agglomerative")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Data dir: /kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data
Results dir: /kaggle/working/results/e5_agglomerative


In [3]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

(750, 6) (750, 8) (800, 6)


In [4]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

df shape: (2000, 6)


,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [5]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

df shape after required-field dropna: (1999, 6)
<class 'pandas.core.frame.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   article_id    1999 non-null   object
 1   publisher     1999 non-null   object
 2   url           1999 non-null   object
 3   published_at  1999 non-null   object
 4   title         1999 non-null   object
 5   body_text     1999 non-null   object
dtypes: object(6)
memory usage: 109.3+ KB


In [6]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [7]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

,title,text
0,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,ස්ථාන දෙකකදී ඝාතන දෙකක්,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [8]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()

df shape after text cleaning: (1999, 7)


,article_id,publisher,url,published_at,title,body_text,text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...,"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [9]:
# Build documents for embedding (title + body), with the "passage: " prefix multilingual-e5
# models require for corpus-side text
def build_passage_text(title, body):
    title = str(title).strip() if title else ""
    body = str(body).strip() if body else ""
    combined = f"{title}. {body}" if title else body
    return "passage: " + combined


df["passage_text"] = df.apply(lambda row: build_passage_text(row["title"], row["text"]), axis=1)
print(f"Documents: {len(df)}")
df["passage_text"].iloc[0][:500]

Documents: 1999


'passage: දැන් තෝරු-මෝරු අහුවෙන කාලේ. දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහනුවර දිස්ත්\u200dරික් මන්ත්\u200dරී ජගත් මනුවර්ණ මහතා:-(ජා.ජ.බ) පාර්ලිමේන්තුවේදී පැවසීය.\nදූෂණයට විරුද්ධ වීම භයානක බවත් දූෂණයට විරුද්ධ නොවී සිටීම ඊට වඩා භයානක බවත් අනුර දිසානායක ජනාධිපති\xa0 එක්සත් ජාතීන්ගේ මහා මණ්ඩලයේ අමතමින් ප්\u200dරකාශ කළා.එය අප නැවත අවධාරණය කළ යුතුයි.අපි දේශපාලන පලි ගැනීම් කරනවා යැයි චෝදනා කරනවා.නමුත් ඇත්ත ඒක නෙමෙයි.මත්තල ගුවන් තොටුපොලේ මගින් පර්යන්තය එක\xa0 ආණ්ඩුවක් නෙළුම් පොහොට්ටුවක හැඩයකට හදන්න තීරණය කළාම ඊට පස'

In [10]:
# Load multilingual-e5-large (the finalized embedding model for this project)
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name).to(device)
model.eval()

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 1024, padding_idx=1)
    (token_type_embeddings): Embedding(1, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 1024, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-23): 24 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwi

In [11]:
# Mean pooling — established as the best strategy for this model on this corpus in
# notebooks/clustering_e5.ipynb (separation_score 0.2130 for mean vs 0.1264 for max pooling)
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_passages(texts, batch_size=16, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


embeddings = embed_passages(df["passage_text"].tolist(), batch_size=16)
print(embeddings.shape)

(1999, 1024)


In [12]:
# Embedding separation score (comparable across notebooks): 1 - mean pairwise cosine similarity
# on a random 100-document sample of the raw embeddings.
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(embeddings), size=min(100, len(embeddings)), replace=False)
sample_emb = embeddings[sample_idx]

sims = cosine_similarity(sample_emb)
pairwise = sims[np.triu_indices_from(sims, k=1)]
separation_score = 1 - pairwise.mean()

print(f"Embedding model: {embedding_model_name}")
print(f"mean_sim={pairwise.mean():.4f} std={pairwise.std():.4f} min={pairwise.min():.4f} max={pairwise.max():.4f}")
print(f"Separation score (1 - mean cosine similarity): {separation_score:.4f}")

Embedding model: intfloat/multilingual-e5-large
mean_sim=0.7909 std=0.0243 min=0.7123 max=0.9162
Separation score (1 - mean cosine similarity): 0.2091


In [13]:
# Reduce dimensionality with UMAP (same settings used across this project's UMAP-based
# notebooks, e.g. notebooks/clustering_BGE_M3_BERTopic.ipynb)
from umap import UMAP

umap_model = UMAP(
    n_neighbors=3,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)
reduced_embeddings = umap_model.fit_transform(embeddings)
print(reduced_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(1999, 5)


In [14]:
# Sweep Agglomerative (Ward) distance_threshold, pick the cut that maximizes silhouette score
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

candidate_thresholds = np.linspace(0.5, 15.0, 30)
sweep_results = []

for threshold in candidate_thresholds:
    model = AgglomerativeClustering(n_clusters=None, distance_threshold=threshold, linkage="ward")
    labels = model.fit_predict(reduced_embeddings)
    n_found = len(set(labels))
    if 1 < n_found < len(labels):
        sil = silhouette_score(reduced_embeddings, labels, metric="cosine")
        sweep_results.append((threshold, n_found, sil))

sweep_df = pd.DataFrame(sweep_results, columns=["distance_threshold", "n_clusters", "silhouette"])
sweep_df = sweep_df.sort_values("silhouette", ascending=False)
sweep_df.head(10)

,distance_threshold,n_clusters,silhouette
0,0.5,330,0.805480
3,2.0,137,0.779719
1,1.0,211,0.771403
2,1.5,162,0.767512
4,2.5,121,0.766601
6,3.5,99,0.752609
5,3.0,107,0.738273
7,4.0,87,0.733187
8,4.5,80,0.723221
9,5.0,78,0.719211


In [15]:
# Refit at the best threshold found
best_threshold = float(sweep_df.iloc[0]["distance_threshold"])
print(f"Best distance_threshold: {best_threshold:.3f} (n_clusters={int(sweep_df.iloc[0]['n_clusters'])})")

model = AgglomerativeClustering(n_clusters=None, distance_threshold=best_threshold, linkage="ward")
df["cluster_id"] = model.fit_predict(reduced_embeddings)

print(df["cluster_id"].value_counts().head(20))
print("Number of clusters found:", df["cluster_id"].nunique())

extra_row_fields = {"distance_threshold": round(best_threshold, 3)}

Best distance_threshold: 0.500 (n_clusters=330)
cluster_id
0      20
1      20
52     16
92     16
54     16
13     14
64     14
23     13
62     13
176    13
35     13
77     12
5      12
47     12
80     12
55     12
29     12
88     12
43     12
63     11
Name: count, dtype: int64
Number of clusters found: 330


In [16]:
# Clustering evaluation
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

labels = df["cluster_id"].values
mask = labels != -1  # -1 = noise; only meaningful for density-based algorithms
X_valid = reduced_embeddings[mask]
labels_valid = labels[mask]
n_clusters = len(set(labels_valid))
noise_ratio = 1 - mask.mean()

print(f"Model: {embedding_model_name} + UMAP + Agglomerative(Ward)")
print(f"Articles: {len(df)} | Clusters (excl. noise): {n_clusters} | Noise ratio: {noise_ratio:.2%}")

if n_clusters > 1:
    sil = silhouette_score(X_valid, labels_valid, metric="cosine")
    dbi = davies_bouldin_score(X_valid, labels_valid)
    ch = calinski_harabasz_score(X_valid, labels_valid)
    print(f"Silhouette Score (cosine): {sil:.4f}")
    print(f"Davies-Bouldin Index:      {dbi:.4f}  (lower is better)")
    print(f"Calinski-Harabasz Index:   {ch:.2f}  (higher is better)")
else:
    sil = dbi = ch = float("nan")
    print("Not enough clusters to compute silhouette/DBI/CH.")

scores_path = RESULTS_DIR / "e5_agglomerative_scores.csv"
row = {
    "model": embedding_model_name,
    "pipeline": "UMAP+Agglomerative(Ward)",
    "embedding_dim": embeddings.shape[1],
    "separation_score": round(separation_score, 4),
    "n_articles": len(df),
    "n_clusters": n_clusters,
    "noise_ratio": round(noise_ratio, 4),
    "silhouette": round(sil, 4) if n_clusters > 1 else None,
    "davies_bouldin": round(dbi, 4) if n_clusters > 1 else None,
    "calinski_harabasz": round(ch, 2) if n_clusters > 1 else None,
}
row.update(extra_row_fields)
pd.DataFrame([row]).to_csv(scores_path, index=False)
print(f"Saved scores to {scores_path}")

Model: intfloat/multilingual-e5-large + UMAP + Agglomerative(Ward)
Articles: 1999 | Clusters (excl. noise): 330 | Noise ratio: 0.00%
Silhouette Score (cosine): 0.8055
Davies-Bouldin Index:      0.4009  (lower is better)
Calinski-Harabasz Index:   12028.33  (higher is better)
Saved scores to /kaggle/working/results/e5_agglomerative/e5_agglomerative_scores.csv


In [17]:
# Inspect sample titles per cluster
for cluster_id, group in list(df[df["cluster_id"] != -1].groupby("cluster_id"))[:10]:
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(10):
        print(f"- {title}")
    print()

=== Cluster 0 (20 articles) ===
- ශ්‍රී ලංකාවේ පාසල් අධ්‍යාපනය: විදුහල්පති තනතුරු පිරිනමන්නේ ජාතිකත්වය හෝ ඇඳුම මත පදනම්ව ද?
- ශ්‍රී ලංකා පොලිසියේ මානව හිම්කම් උල්ලංඝණ ලැයිස්තුව තවත් දිගු වෙයි
- පොලිස් නිලධාරියකු සිවිල් ඇඳුමින් කරන ක්‍රියාවන්හි වගකීමෙන් රජයට බැහැර විය නොහැකි බවට ශ්‍රේෂ්ඨාධිකරණයෙන් තීන්දුවක්
- මුල්ලිවයික්කාල් කැඳ දන්සැල් තහනම් කිරීම සහ අත්අඩංගුවට ගැනීම් සිදු කරන්නේ ඇයි?
- මානව හිමිකම් කොමිසම සිය මානව හිමිකම් කඩකළ බවට එහි නිලධාරිනියක් අඛණ්ඩ සත්‍යග්‍රහයක නිරත වූයේ ඇයි?
- රංගන ශිල්පියෙකුගේ රංගන ශෛලිය අනුකරණය කිරීම අපරාධමය වරදක්ද?
- ශ්‍රී ලංකා පොලිසියේ මානව හිමිකම් උල්ලංඝන: වින්දිතයින්ට කෝටි 15ක් වන්දි ගෙවන්නට සිදු වූ පොලිස් නිලධාරීන් කවුද?
- ජන අරගල ව්‍යාපාරයේ විරෝධතාවයේ දී 29ක් පොලිස් අත්අඩංගුවට
- අණ නොතකා වාහනය පදවන්නන්ට උරුම වෙඩි කා මිය යාම ද?
- චුන්නාකම් පොලීසියේ 4කගේ වැඩ තහනම්

=== Cluster 1 (20 articles) ===
- ජන අරගල සන්ධානය : 'ගෝල්ෆේස් අරගලකරුවන්' රැසක් සමග දේශපාලන කරලියට එන නව සන්ධානය
- මහ මැතිවරණයක් පැවැත්වීමට ආණ්ඩුවට මුදල් තිබේ ද?
- රනිල් වික්‍රමසිංහ:'ත්‍රී සිංහල

In [18]:
# Inspect a RANDOM sample of clusters (rather than just the first few by id) — a more
# representative check of overall cluster quality than always looking at the same low-numbered
# clusters
rng_inspect = np.random.default_rng(SEED)
cluster_ids = df.loc[df["cluster_id"] != -1, "cluster_id"].unique()
sample_size = min(8, len(cluster_ids))
sampled_cluster_ids = rng_inspect.choice(cluster_ids, size=sample_size, replace=False)

for cluster_id in sampled_cluster_ids:
    group = df[df["cluster_id"] == cluster_id]
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

=== Cluster 301 (3 articles) ===
- යාපනයේ සුපිරි පන්දුයවන්නා ධෝනි හමුවෙයි
- මංගල තරගයේදී ම පිත්තෙන් වාර්තා තැබූ පන්දු යවන්නා
- LPL 2024 : පළමු තරගයේදීම අර්ධ ශතකයක් වාර්තා කළ චමිදු වික්‍රමසිංහ යනු කවුද?

=== Cluster 52 (16 articles) ===
- දෙහි කිලෝව රු. 2000යි, ගෙඩියක් රු 120යි
- දවල්ට අව්ව - රෑට අලි, පීඩාව දරාගෙන වැවූ අස්වැන්නට මිලක් නැතිව රජරට ගොවියෝ ලතැවෙති
- පුවක් ගෙඩියක මිල රුපියල් 20ත් 25 අතරට  යයි
- නාඩු සහල් හිඟයක්
- කීරී සම්බා හිඟයට හේතුව ඩඩ්ලි කියයි
- ටින් මාළු සඳහා පාලන මිලක්
- ආනයනික භාණ්ඩ කිහිපයක මිල පහළට
- කොළඹ දෙහි කිලෝව 2000යි රජරට දෙහි විකුණගන්නත් බෑ
- බීළූණු බීජ කිලෝ එකක් රුපියල් 35000ක්
- සහන මිලට පොල් හා පොල් නිෂ්පාදන ගන්න තැනක්
- සතොස භාණ්ඩ වර්ග තුනක මිල අඩු කරයි
- මේ වසරේදී පොල් අස්වැන්න අඩුවීමට හේතුව මෙන්න
- නාඩු කිලෝ 5000ක් ගන්න කීරි සම්බා කිලෝ 15000ක් මිලදී ගන්න කියයි
- මස් හා මාළු මිල ඉහළට
- රජය පොලොන්නරුවේ ගොවීන්ගේ වී මිලදී ගැනීම අරඹයි

=== Cluster 133 (9 articles) ===
- හිරු එළියෙන් සිතුවම් මවන මඩකලපුවේ ශිවනේෂරාසා
- මධ්‍යම අප්‍රිකානු සෙබළුන් 14ක් බේරාගත් ලංකා

In [19]:
# Save article-level assignments
assignments_path = RESULTS_DIR / "e5_agglomerative_assignments.csv"
df.drop(columns=["passage_text"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
print(f"Saved assignments: {assignments_path.resolve()}")

Saved assignments: /kaggle/working/results/e5_agglomerative/e5_agglomerative_assignments.csv
